<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_26_low_high_frequency_special_regimes.ipynb">9.26 低频与高频特殊观测体制</a></li>
        <li>下一节： <a href="9_x_further_reading_and_workflow.ipynb">9.x 延伸阅读与后续实践方向</a></li>
    </ul>
</div>


## 9.27 Pipeline QA、Weblog 与再处理决策：从自动产品到可审查结果

现代射电观测越来越依赖自动 pipeline。VLA、ALMA、MeerKAT、LOFAR、VLBI 和许多台站项目都会给出不同层次的自动校准、成像产品、QA 报告、weblog 或 pipeline summary。自动产品极大降低了入门门槛，也让归档数据再分析成为常规研究路径；但它们同时带来一个新的训练问题：pipeline 标记为完成的产品，并不自动等于适合某个新的科学问题。

Pipeline 的核心作用，是在一组预设假设下把原始数据转化为标准产品。它会选择 flag 策略、参考天线、校准源模型、解区间、权重、成像参数、mask、阈值和质量门。多数情况下，这些默认选择是合理的；但当科学目标变成低面亮度结构、弱谱线翼、偏振角、主波束边缘源、短时间变源、毫米波弱线或高动态范围成像时，pipeline 的“通过”只说明标准流程没有发现致命错误，不说明新的科学量已经无偏、足够深或误差预算完整。

这一节把第 9.23 节的归档数据判断、第 9.24 节的 provenance、第 9.26 节的天气和特殊频段 QA 连接起来。目标是建立一种读 weblog 的教材方法：先看观测和 pipeline 假设，再看 flag 和校准证据，再看成像和科学区域诊断，最后决定直接使用、重新成像、部分重校准、完整重处理，还是放弃该数据集。


### 9.27.1 Pipeline 产品是一组带假设的中间结论

Pipeline 输出通常包括校准后的 Measurement Set、校准表、flag 版本、图像、cube、残差、mask、源表、QA 图和 weblog。它们看起来像最终结果，但从测量链角度看，它们只是中间结论：在某些数据被 flag、某些模型被采用、某些权重被定义、某些成像参数被固定后，得到了一组最可能适用于常规目标的产品。

因此，pipeline 产品的使用必须回到目标科学量。若目标是一个强点源的总通量，标准连续谱 pipeline 的图像和通量标尺可能已经足够；若目标是主波束半功率点之外的弱源谱指数，就必须重新检查宽带主波束、频率相关 uv 覆盖和局部噪声；若目标是弱谱线吸收，bandpass 残差、线外通道、Hanning 平滑、RFI flag 和 continuum subtraction 的细节比最终图像是否美观更重要。

可以把 pipeline 产品的科学可用性写成一个误差不等式。设目标科学量为 $q$，pipeline 产物给出的估计为 $\hat q$，要求的不确定度和系统偏差容忍度分别为 $\sigma_{\rm req}$ 与 $b_{\rm req}$。产品可用不只是要求图像 RMS 小于某个值，而是要求

$$
\sigma(\hat q)\lesssim \sigma_{\rm req},\qquad
|\mathrm{bias}(\hat q)|\lesssim b_{\rm req}.
$$

Weblog 的价值就在于提供判断这两个量所需的证据：哪些数据被删掉，校准解是否稳定，天气和系统温度是否异常，成像残差是否有结构，pipeline mask 是否适合科学区域，产品历史是否足以复现。


![pipeline weblog anatomy](figures/practical_pipeline_weblog_anatomy.png)

图中把 pipeline weblog 看成一张证据地图。中间是从原始数据到产品的处理链，上下方是支撑判断的证据：元数据、flag 摘要、校准解、天气或系统温度、图像 QA、警告、参数版本和科学目标匹配。有效的 weblog 不只是给出 pass/fail，而是把每个产品和证据连起来。


### 9.27.2 Weblog 的基本解剖：先读数据身份，再读图像

读 weblog 的第一步应是确认数据身份和观测结构。项目编号、观测日期、execution block、阵列配置、频率设置、spectral window、通道宽度、校准源、on-source 时间、天气条件和 pipeline 版本共同决定了产品能否回答新的科学问题。若这些元数据已经不匹配，后续图像再漂亮也不能补足缺失信息。

第二步是检查 flag 与权重。若 flag 在时间、频率、天线或扫描上接近随机分布，热噪声可近似按有效数据量变化。若随机 flag 比例为 $f$，热噪声量级会从 $\sigma$ 变成

$$
\sigma'\simeq \frac{\sigma}{\sqrt{1-f}}.
$$

但真实 flag 很少完全随机。若一个短基线天线被大量 flag，最大可恢复尺度和表面亮度灵敏度会比这个公式更差；若某个 spectral window 的边缘或窄频段被严重 flag，谱线覆盖、RM synthesis 和宽带谱指数都会受到结构化影响。Weblog 中的 flag summary 应被当作 uv 覆盖、频率覆盖和校准约束的变化图，而不是简单百分比。

第三步是检查校准证据。Bandpass 应当在无 RFI、足够信噪比的频段内平滑；gain phase 应当在解区间内有合理连续性；振幅解不应在 execution block 之间无解释跳变；flux calibrator 模型和 transfer 应当与频段匹配；参考天线切换、bad antenna、failed solution、低 S/N 解和未应用的校准表都应被追踪。最后才进入图像产品，检查 beam、RMS、残差、mask、主波束、权重和科学区域。


### 9.27.3 QA 等级必须按科学目标重新加权

许多 pipeline 会给出类似 PASS、WARNING、FAIL 或不同 QA 等级的标记。这些标记有用，但不能直接替代科学判断。一个对普通连续谱点源足够的 QA 状态，对偏振、弱谱线、宽场或高频项目可能只是最低入口。原因是 pipeline QA 的默认目标通常是“标准产品是否整体合理”，而研究问题关心的是“某个科学区域、某个频率范围、某个 Stokes 参数或某个尺度的测量是否可信”。

例如，metadata 对强点源连续谱可能不是主要风险，但对谱线红移、速度约定或多配置组合至关重要。Calibrator solution 对普通 Stokes I 通量可能只是常规检查，但对偏振泄漏和绝对偏振角会变成核心风险。Weather/Tsys 对低频连续谱可能影响较小，却可能决定毫米波弱线是否可用。Image residual 对中心紧致源可能可接受，但对宽场主波束边缘和低面亮度结构可能完全不能接受。

因此，weblog 判读应把 QA 证据和科学目标组成矩阵。每一项证据不是独立打分，而是按它对目标科学量的影响重新加权。这样的训练能避免两种常见错误：一是看到 pipeline PASS 就直接测量，二是看到某个 WARNING 就完全放弃数据。正确的做法是追问 WARNING 影响的是哪条数据轴、哪个科学区域和哪一项误差预算。


![pipeline QA evidence matrix](figures/practical_pipeline_qa_evidence_matrix.png)

图中用矩阵示意 QA 证据如何随科学目标改变权重。同样的 flag、calibrator solution 或 weather/Tsys 信息，对连续谱通量、谱线、偏振、宽场和高频观测的重要性不同。绿色表示通常足够，黄色表示需要读图和日志，红色表示直接进入科学误差预算。


### 9.27.4 校准日志中的关键线索

校准日志和诊断图应一起读。日志中的 `WARN` 或 `ERROR` 只是指针，真正的判断来自它指向的数据轴和物理对象。若日志提示某个天线 flag fraction 高，需要检查包含该天线的基线是否集中在短基线或长基线、是否影响校准源扫描、是否改变目标 $uv$ 覆盖。若日志提示某个 spectral window bandpass 不稳定，需要检查该窗口是否包含目标线、是否只在边缘通道异常、是否有 RFI 或低 S/N 问题。若日志提示相位跳变，需要判断它发生在校准源、目标源还是特定 execution block。

校准解的信噪比也可用数量级估计帮助判读。对一个校准源，单基线热噪声近似由 SEFD、带宽和解区间决定；若校准源通量为 $S_{\rm cal}$，解的有效信噪比大致随

$$
\mathrm{SNR}_{\rm sol}\propto S_{\rm cal}\sqrt{\Delta\nu\,\tau}
$$

增长。过短解区间会降低 S/N，过长解区间会平均掉真实相位变化。Pipeline 的默认解区间往往是一种折中；在高频、低频电离层活跃、弱校准源或快速变源项目中，这个折中可能需要重新评估。

图像日志同样需要物理解释。自动 mask 失败、清洁深度不足、达到 iteration limit、模型发散、残差非高斯、主波束边缘噪声升高，都不是简单的成像参数问题。它们可能来自上游 flag、校准、权重、缺短间距、方向相关效应或源模型不完备。Pipeline weblog 的教学价值正是在这里：它把“图像不好”拆回到可追踪的处理步骤。


![pipeline calibration log diagnostics](figures/practical_pipeline_calibration_log_diagnostics.png)

图中把几类典型日志线索和诊断图放在一起。相位跳变、窄频 bandpass 异常、天线级 flag 集中和日志 warning/error 需要互相指认。日志不是结论，而是指向时间、频率、天线、扫描或 spectral window 的线索。


### 9.27.5 再处理决策：直接用、重成像、重校准还是放弃

归档数据再分析常见的错误，是在没有读完 weblog 的情况下直接完整重处理。完整重处理并不总是更科学：它可能丢掉 pipeline 中已经验证过的校准表，也可能引入新的人工选择。更稳健的做法是先按科学问题决定最小必要修改。

若 pipeline 产品的元数据、校准、噪声、beam、残差和科学区域都满足要求，可以直接使用产品进行测量，但必须记录项目编号、pipeline 版本、产品类型和验证结果。若校准表可靠而图像参数不匹配，例如 cell size、weighting、taper、mask、threshold 或 spectral cube 轴设置不合适，可以保留校准结果，只重新成像或重新加权。若 flag、bandpass、gain、flux scale、Tsys、WVR 或方向相关解存在科学相关问题，则需要部分或完整重校准。若关键观测设置缺失，例如没有目标频率、没有足够短基线、校准源不满足偏振或高频要求、天气完全不可恢复，则应明确说明该归档数据不能回答当前问题。

这个决策过程应被写入处理记录。科学结果不应只报告“使用 pipeline image”或“手动 reprocess”，而应说明为什么选择这个分支：哪些证据支持直接使用，哪些证据要求重成像，哪些证据迫使重校准，哪些限制不能通过后处理弥补。


![pipeline reprocessing decision tree](figures/practical_pipeline_reprocessing_decision_tree.png)

图中给出归档 pipeline 产品的再处理决策树。第一判断不是产品是否存在，而是科学目标和元数据、weblog 是否匹配。之后才决定直接测量、重新成像、部分或完整重校准，或判断数据集不适合该问题。


### 9.27.6 产品级验证：把图像和 weblog 重新闭合

无论是否重新处理，最终产品都需要独立验证。图像 RMS 应与权重、带宽、on-source 时间和 flag 比例在数量级上一致；若实测 RMS 远高于热噪声预期，应检查动态范围限制、残差结构、RFI、校准误差和混淆噪声。Restored beam 应与科学角尺度匹配；若 beam 过大，位置和结构解释受限；若 beam 过小，低面亮度结构可能被分裂成低信噪比像素。主波束校正会放大噪声，因此科学区域应明确限定到主波束模型和局部 RMS 可信的范围。

Spectral cube 的验证还要逐 spectral window、逐通道或逐速度范围进行。单个全局 RMS 不能代表边缘通道、水汽吸收附近、高 flag 通道或强 RFI 残留频段。Moment map、PV 图和谱线积分通量必须和 mask、channel noise、beam、continuum subtraction 和速度约定一起报告。偏振产品则必须检查 leakage、cross-hand phase、偏振角标尺、RM synthesis 的 RMSF 和 off-axis polarization。低频、高频和 VLBI 产品还需要分别回到电离层、天气相干性和 fringe fitting 诊断。

这种验证不是对 pipeline 的不信任，而是把 pipeline 产品纳入可审查的科学链。自动处理负责给出标准起点，人工验证负责说明标准起点是否适合当前目标。


### 9.27.7 与真实软件和台站产品的对应

VLA CASA pipeline 的 weblog 通常包含 import、flagging、calibration、applycal、plotms 风格诊断、image products 和 QA 信息。ALMA pipeline weblog 更强调 execution block、weather/PWV、$T_{\rm sys}$、WVR、bandpass、gain、flux、calibration strategy、QA2 状态和 science goal products。低频 pipeline 可能以 AOFlagger、DP3、DDFacet、WSClean 或台站专用 summary 的形式呈现，重点是 RFI、方向相关解、beam 模型、facet 残差和源表一致性。VLBI 处理则可能更依赖 fringe report、delay/rate 解、参考天线、SEFD、$T_{\rm sys}$、gain curve 和相位参考诊断。

这些软件生态不同，但判读对象相似：数据身份、flag 版本、校准表、模型、权重、成像参数、日志、诊断图、人工修改和最终验证。一个可复现项目应把 pipeline 原始记录和手动修改记录一起保存。若只保存最终 FITS 图，无法解释与原 pipeline 产品的差异；若只保存手动脚本而不保存原 weblog，也无法判断手动处理修复了什么问题。


![pipeline reprocessing provenance bundle](figures/practical_pipeline_reprocessing_provenance_bundle.png)

图中把再处理结果需要保存的 provenance 拆成六类：原始数据身份、pipeline 记录、手动修改、配置文件、中间产品和验证结果。归档再处理的可复现性来自 pipeline provenance 与 manual provenance 的合并，而不是只保存最后图像。


### 9.27.8 教学案例：一个 pipeline PASS 的弱谱线项目

设想一个归档数据集，原项目目标是宽带连续谱成像，pipeline weblog 给出 PASS，中心场连续谱图像 RMS 接近预期。新的科学目标却是寻找目标附近一条弱谱线吸收，吸收深度只有连续谱峰值的千分之几，位于一个 RFI 较多的 spectral window 边缘。若只看 pipeline 总体状态，这个数据集似乎可用；若按科学目标读 weblog，风险会很快暴露。

首先，metadata 需要确认 spectral window 是否覆盖目标速度范围，通道宽度和 Hanning smoothing 是否足够解析吸收线。其次，flag summary 需要检查目标线附近通道是否被大量 flag，RFI 是否集中在相邻通道，bandpass calibrator 是否在该 spectral window 有足够 S/N。随后，bandpass 解和校准后校准源谱线应检查是否存在宽浅波纹；若 pipeline 成像只生成 continuum MFS 图，就必须重新生成 cube，并在 uv 域或图像域做适合谱线的 continuum subtraction。最后，目标谱线的显著性不能只用全频段 RMS，而要用线附近的局部通道噪声、残余 continuum ripple、mask 选择和空白区域假线率评估。

这个案例展示了 pipeline PASS 的正确用法。PASS 说明观测和标准连续谱产品没有明显失败；它不说明弱谱线科学已经完成。合理结论可能是：保留 pipeline 的基础校准表，重新检查 line spectral window 的 flag 和 bandpass，重新做 continuum subtraction 与 cube imaging，并把 line-free 通道噪声和注入线恢复作为验证。若 weblog 显示目标线所在频段被严重 RFI 覆盖，则应报告该归档数据对弱吸收线目标不具备足够约束。


### 9.27.9 本节结论

Pipeline、weblog 和 QA 报告应被理解为证据系统，而不是自动科学结论。它们记录了数据身份、flag、校准、天气、系统温度、成像参数、警告和产品历史；这些信息只有和具体科学目标相结合，才能判断产品是否可用。直接使用、重新成像、部分重校准、完整重处理或放弃数据，都是合理分支，关键是每个分支都必须由证据支持。

第 9.27 节把第 9 章后半部分进一步闭合：观测设计说明数据如何产生，软件生态说明结果如何被记录，VLBI 与特殊频段说明主导误差如何改变，而 pipeline/weblog 判读则说明已有自动产品如何进入可审查的科学测量。后续若继续加厚，可加入真实 VLA/ALMA/LOFAR weblog 页面截图式案例、失败 pipeline 产品复盘和最小再处理项目模板。
